# Round 1 - Phase 2: Analytical Core

**Competition:** Data Vortex | AARUUSH'26
**Theme:** Rebuilding the Social Engine
**Team:** Entropy
**Members:** Saanvi Grover & Aditya Sharma

Questions solved: **E3** (Average Engagement by Platform), **M4** (Platform Behaviour by High-Follower Users), **H4** (Follower-to-Engagement Anomaly).

This notebook (1) builds the SQLite schema from the Phase 1 cleaned CSVs with DDL-level constraints, (2) shows the query plan for each query so the indexes are visibly used, (3) runs the three final queries stored in `queries/`, and (4) reproduces every statistic quoted in the logic explanation and insight report.

In [1]:
import sqlite3
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 20)

DB = "Entropy_social_engine.db"
P1 = Path("../Round1-Phase1-Data-Recovery")
posts = pd.read_csv(P1 / "Entropy_Social_Engine_Posts_Cleaned.csv")
users = pd.read_csv(P1 / "Entropy_Social_Engine_Users_Cleaned.csv")
print(posts.shape, users.shape)

(12000, 8) (1500, 5)


## 1. Schema

Two normalised tables. `posts.user_id` is a foreign key to `users`. NOT NULL and CHECK constraints encode what Phase 1 cleaning guarantees; the three corrupted columns (`platform`, `text_content`, `likes`) stay nullable because Phase 1 showed the corruption is Missing Completely at Random.

Indexes serve these queries, and the `EXPLAIN QUERY PLAN` output below each query confirms it: `posts(platform)` drives the platform grouping in E3 and M4, `posts(user_id)` drives every posts-to-users join and the per-user roll-up in H4, and the `users` primary key resolves each join lookup. Data is loaded with `if_exists="append"` so the hand-written DDL survives.

In [2]:
DDL = '''
PRAGMA foreign_keys = ON;
DROP VIEW  IF EXISTS user_posts;
DROP TABLE IF EXISTS posts;
DROP TABLE IF EXISTS users;

CREATE TABLE users (
    user_id         TEXT PRIMARY KEY,
    location        TEXT NOT NULL,
    language        TEXT NOT NULL,
    account_created DATE NOT NULL,
    follower_count  INTEGER NOT NULL CHECK (follower_count >= 0)
);

CREATE TABLE posts (
    post_id      TEXT PRIMARY KEY,
    user_id      TEXT NOT NULL REFERENCES users(user_id),
    platform     TEXT,
    text_content TEXT,
    timestamp    DATETIME NOT NULL,
    likes        INTEGER CHECK (likes >= 0),
    shares       INTEGER NOT NULL CHECK (shares >= 0),
    comments     INTEGER NOT NULL CHECK (comments >= 0)
);

CREATE INDEX idx_posts_user_id         ON posts(user_id);
CREATE INDEX idx_posts_platform        ON posts(platform);
CREATE INDEX idx_posts_timestamp       ON posts(timestamp);
CREATE INDEX idx_users_follower_count  ON users(follower_count);
CREATE INDEX idx_users_location        ON users(location);
CREATE INDEX idx_users_language        ON users(language);

CREATE VIEW user_posts AS
SELECT p.*, u.location, u.language, u.account_created, u.follower_count,
       p.likes + p.shares + p.comments AS total_engagement
FROM posts p JOIN users u ON u.user_id = p.user_id;
'''
con = sqlite3.connect(DB)
con.executescript(DDL)
users.to_sql("users", con, if_exists="append", index=False)
posts.to_sql("posts", con, if_exists="append", index=False)
con.commit()
print(pd.read_sql_query("SELECT type, name FROM sqlite_master WHERE type IN ('table','index','view') AND name NOT LIKE 'sqlite_%' ORDER BY type, name", con))

    type                      name
0  index        idx_posts_platform
1  index       idx_posts_timestamp
2  index         idx_posts_user_id
3  index  idx_users_follower_count
4  index        idx_users_language
5  index        idx_users_location
6  table                     posts
7  table                     users
8   view                user_posts


In [3]:
# Constraints are enforced by the engine, not just by pandas
checks = {
    "foreign key": "INSERT INTO posts VALUES ('x1','no_such_user','Reddit',NULL,'2025-01-01',1,1,1)",
    "CHECK likes >= 0": "INSERT INTO posts VALUES ('x2',(SELECT user_id FROM users LIMIT 1),'Reddit',NULL,'2025-01-01',-5,1,1)",
    "NOT NULL shares": "INSERT INTO posts VALUES ('x3',(SELECT user_id FROM users LIMIT 1),'Reddit',NULL,'2025-01-01',1,NULL,1)",
}
for name, sql in checks.items():
    try:
        con.execute(sql)
        print(f"{name}: NOT enforced")
    except sqlite3.IntegrityError as e:
        print(f"{name}: rejected ({e})")
con.rollback()

foreign key: rejected (FOREIGN KEY constraint failed)
CHECK likes >= 0: rejected (CHECK constraint failed: likes >= 0)
NOT NULL shares: rejected (NOT NULL constraint failed: posts.shares)


In [4]:
def load(key):
    names = {"E3": "E3_average_engagement_by_platform",
             "M4": "M4_platform_behaviour_high_follower_users",
             "H4": "H4_follower_to_engagement_anomaly"}
    return Path(f"queries/Entropy_{names[key]}.sql").read_text(encoding="utf-8")

def plan(key):
    return pd.read_sql_query("EXPLAIN QUERY PLAN " + load(key), con)[["detail"]]

def holm(pvals):
    p = np.asarray(pvals, dtype=float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0.0
    for i, idx in enumerate(order):
        running = max(running, (len(p) - i) * p[idx])
        adj[idx] = min(running, 1.0)
    return adj

## 2. E3 - Average Engagement by Platform (Easy)

**Challenge:** Calculate the average likes, shares and comments for each platform. Which platform generates the highest average total engagement?

**Design:** only posts with a missing platform are excluded, as the question implies. `AVG()` skips NULLs, so average likes uses posts that have a like count while shares and comments use every post. For total engagement, `likes + shares + comments` is NULL when likes is missing, so `AVG()` ignores those posts, which is exactly the brief's definition of total engagement (E2). A cross-joined grand mean adds `pct_vs_overall`.

In [5]:
plan('E3')

,detail
0,CO-ROUTINE (subquery-4)
1,CO-ROUTINE (subquery-5)
2,CO-ROUTINE platform_stats
3,SEARCH posts USING INDEX idx_posts_platform (p...
4,MATERIALIZE overall
5,SCAN posts
6,SCAN ps
7,SCAN o
8,USE TEMP B-TREE FOR ORDER BY
9,SCAN (subquery-5)


In [6]:
e3 = pd.read_sql_query(load("E3"), con)
e3

,engagement_rank,platform,posts,posts_with_likes,avg_likes,avg_shares,avg_comments,avg_total_engagement,pct_vs_overall
0,1,Instagram,1989,1693,2500.9,1040.8,499.8,4040.0,0.78
1,2,YouTube,2073,1747,2517.8,1011.8,504.4,4031.8,0.58
2,3,Facebook,2074,1755,2528.9,984.2,506.9,4016.0,0.18
3,4,Reddit,2031,1742,2488.1,1002.2,511.2,4003.9,-0.12
4,5,Twitter,2049,1725,2437.7,1005.4,506.1,3951.9,-1.42


In [7]:
ep = pd.read_sql_query('''
    SELECT platform, likes + shares + comments AS eng
    FROM posts WHERE platform IS NOT NULL AND likes IS NOT NULL''', con)
kw = stats.kruskal(*[g.eng.values for _, g in ep.groupby("platform")])
sd, n_per = ep.eng.std(), ep.groupby("platform").size().mean()
mde = 2.8 * sd * np.sqrt(2 / n_per)   # 80% power, two-sided alpha = 0.05
print(f"Kruskal-Wallis across platforms: H = {kw.statistic:.2f}, p = {kw.pvalue:.3f}")
print(f"First-to-last spread: {100*(e3.avg_total_engagement.max()-e3.avg_total_engagement.min())/ep.eng.mean():.2f}% of the mean")
print(f"Minimum detectable difference between two platforms: {mde:.0f} ({100*mde/ep.eng.mean():.1f}% of the mean)")

Kruskal-Wallis across platforms: H = 2.97, p = 0.562
First-to-last spread: 2.20% of the mean
Minimum detectable difference between two platforms: 150 (3.7% of the mean)


**What this means:** Instagram is first (4,040.0) and Twitter last (3,951.9), a 2.2% spread with p = 0.56. With about 1,700 posts per platform the test could detect a gap of about 3.7%, so this is not a sample-size problem: any real platform effect is smaller than that.

## 3. M4 - Platform Behaviour by High-Follower Users (Medium)

**Challenge:** Among users with at least 30,000 followers, which platform gives them the highest average engagement per post?

**Design:** the cohort is compared with users **below** 30,000 followers. Comparing with all users would put the cohort on both sides of the comparison and shrink any gap. Conditional aggregation computes both groups in one scan; ranking each group separately tests whether the platform order is stable.

In [8]:
plan('M4')

,detail
0,CO-ROUTINE (subquery-4)
1,CO-ROUTINE (subquery-5)
2,CO-ROUTINE (subquery-6)
3,CO-ROUTINE by_platform
4,SEARCH p USING INDEX idx_posts_platform (platf...
5,SEARCH u USING INDEX sqlite_autoindex_users_1 ...
6,USE TEMP B-TREE FOR count(DISTINCT)
7,SCAN by_platform
8,USE TEMP B-TREE FOR ORDER BY
9,SCAN (subquery-6)


In [9]:
m4 = pd.read_sql_query(load("M4"), con)
m4

,rank_high_followers,rank_other_users,platform,high_follower_users,high_follower_posts,avg_eng_high_followers,avg_eng_other_users,lift_pct
0,1,4,Instagram,404,678,4145.2,3969.7,4.42
1,2,2,Facebook,404,674,4000.9,4025.4,-0.61
2,3,3,Reddit,408,677,3981.4,4018.3,-0.92
3,4,5,Twitter,394,664,3978.5,3935.2,1.10
4,5,1,YouTube,419,715,3947.4,4090.4,-3.50


In [10]:
pe = pd.read_sql_query('''
    SELECT p.platform, u.follower_count >= 30000 AS hi, p.likes + p.shares + p.comments AS eng
    FROM posts p JOIN users u ON u.user_id = p.user_id
    WHERE p.platform IS NOT NULL AND p.likes IS NOT NULL''', con)
rows = []
for platform, g in pe.groupby("platform"):
    rows.append((platform, stats.mannwhitneyu(g[g.hi == 1].eng, g[g.hi == 0].eng).pvalue))
tests = pd.DataFrame(rows, columns=["platform", "p_raw"])
tests["p_holm"] = holm(tests.p_raw)
print(tests.round(3).to_string(index=False))
print(f"Cohort vs other users, all platforms: Mann-Whitney p = {stats.mannwhitneyu(pe[pe.hi==1].eng, pe[pe.hi==0].eng).pvalue:.3f}")
print(f"Platform differences within the cohort: Kruskal-Wallis p = {stats.kruskal(*[g.eng for _, g in pe[pe.hi==1].groupby('platform')]).pvalue:.3f}")
print(f"Rank agreement between the two groups: Spearman rho = {stats.spearmanr(m4.rank_high_followers, m4.rank_other_users)[0]:.2f}")

 platform  p_raw  p_holm
 Facebook  0.676   1.000
Instagram  0.023   0.116
   Reddit  0.536   1.000
  Twitter  0.455   1.000
  YouTube  0.061   0.245
Cohort vs other users, all platforms: Mann-Whitney p = 0.982
Platform differences within the cohort: Kruskal-Wallis p = 0.183
Rank agreement between the two groups: Spearman rho = -0.30


**What this means:** Instagram tops the cohort (4,145.2, +4.4% over smaller accounts, raw p = 0.023) but the effect does not survive correction for five platform tests. The platform order reshuffles between groups (YouTube is 1st for smaller accounts and 5th for the cohort), and overall the cohort is indistinguishable from everyone else. A platform recommendation for large accounts would be fitting noise.

## 4. H4 - Follower-to-Engagement Anomaly (Hard)

**Challenge:** Find users with fewer than 5,000 followers whose total post engagement places them in the top 10% of all users.

**Design:** four levels. (1) A missing like count is imputed with that user's own average likes (global average if they have none), so corruption neither erases a post's valid shares and comments nor penalises the users it happened to hit most. (2) Posts roll up to users. (3) Every user is decile-ranked twice, by total and by per-post engagement, with `user_id` as a deterministic tie-break. (4) Survivors of the top-total-decile and follower filters are classified as volume-driven or exceptional per post.

In [11]:
plan('H4')

,detail
0,CO-ROUTINE (subquery-7)
1,CO-ROUTINE ranked_users
2,CO-ROUTINE (subquery-8)
3,CO-ROUTINE (subquery-9)
4,CO-ROUTINE (subquery-10)
5,CO-ROUTINE user_engagement
6,CO-ROUTINE user_like_avg
7,SCAN posts USING INDEX idx_posts_user_id
8,MATERIALIZE global_like_avg
9,SCAN posts


In [12]:
h4 = pd.read_sql_query(load("H4"), con)
h4

,anomaly_rank,user_id,location,follower_count,post_count,imputed_posts,total_engagement,avg_eng_per_post,top_pct_of_users,per_post_decile,classification
0,1,user_uerv85na,"Rome, Italy",1824,16,3,73544.0,4596.5,0.20,2,Volume-driven
1,2,user_hdas0iau,"Rio de Janeiro, Brazil",1620,16,2,63346.0,3959.1,1.07,6,Volume-driven
2,3,user_fgjkkrie,"Lyon, France",2211,14,0,62690.0,4477.9,1.13,3,Volume-driven
3,4,user_n0ok02rt,"Dubai, UAE",2531,18,4,60876.0,3382.0,1.53,9,Volume-driven
4,5,user_6wra58f7,"Johannesburg, South Africa",4953,14,1,57314.0,4093.8,3.54,5,Volume-driven
5,6,user_5oe5t3js,"London, UK",3151,13,2,56063.0,4312.5,3.94,4,Volume-driven
6,7,user_ogtvuuki,"Cairo, Egypt",4459,11,3,54865.0,4987.7,4.60,1,Exceptional per post
7,8,user_r7eg1rac,"Houston, USA",2052,12,0,54552.0,4546.0,4.74,2,Volume-driven
8,9,user_rr1uzkql,"Milan, Italy",1069,14,2,54495.0,3892.5,4.80,6,Volume-driven
9,10,user_67hyf45u,"Vancouver, Canada",898,11,1,52673.0,4788.4,6.14,2,Volume-driven


In [13]:
ue = pd.read_sql_query('''
    WITH ula AS (SELECT user_id, AVG(likes) a FROM posts GROUP BY user_id),
         g   AS (SELECT AVG(likes) a FROM posts),
         pe  AS (SELECT p.user_id, COALESCE(p.likes, ula.a, g.a) + p.shares + p.comments e
                 FROM posts p JOIN ula ON ula.user_id = p.user_id CROSS JOIN g)
    SELECT u.user_id, u.follower_count, COUNT(*) posts, SUM(e) total, AVG(e) per_post
    FROM users u JOIN pe ON pe.user_id = u.user_id GROUP BY u.user_id''', con)
ue = ue.sort_values(["total", "user_id"], ascending=[False, True]).reset_index(drop=True)
ue["total_decile"] = ue.index * 10 // len(ue) + 1
ue = ue.sort_values(["per_post", "user_id"], ascending=[False, True]).reset_index(drop=True)
ue["per_post_decile"] = ue.index * 10 // len(ue) + 1

top = ue[ue.total_decile == 1]
low = ue.follower_count < 5000
observed = int((low & (ue.total_decile == 1)).sum())
expected = len(top) * low.mean()
print(f"Top decile: {len(top)} users, threshold {top.total.min():,.0f}")
print(f"Users with < 5k followers: {low.sum()} ({100*low.mean():.2f}%)")
print(f"Expected anomalies if followers don't matter: {expected:.1f} | observed: {observed} | binomial p = {stats.binomtest(observed, len(top), low.mean()).pvalue:.2f}")
print(f"Exceptional per post among anomalies: {(h4.per_post_decile == 1).sum()} of {len(h4)}; "
      f"share across whole top decile: {100*(top.per_post_decile == 1).mean():.1f}%")
print(f"Mean posts: population {ue.posts.mean():.2f} | top decile {top.posts.mean():.2f} | anomalies {h4.post_count.mean():.2f}")
print(f"Spearman posts vs total:     rho = {stats.spearmanr(ue.posts, ue.total)[0]:.2f}")
print(f"Spearman followers vs total: rho = {stats.spearmanr(ue.follower_count, ue.total)[0]:.3f}, p = {stats.spearmanr(ue.follower_count, ue.total)[1]:.2f}")

Top decile: 150 users, threshold 48,159
Users with < 5k followers: 131 (8.73%)
Expected anomalies if followers don't matter: 13.1 | observed: 16 | binomial p = 0.38
Exceptional per post among anomalies: 2 of 16; share across whole top decile: 15.3%
Mean posts: population 8.00 | top decile 12.85 | anomalies 12.88
Spearman posts vs total:     rho = 0.91
Spearman followers vs total: rho = -0.012, p = 0.64


**What this means:** 16 low-follower users reach the top decile, close to the 13 that chance predicts if followers are irrelevant. 14 of them are there by volume (they post far more than average at ordinary per-post rates). Only 2, `user_ogtvuuki` (Cairo) and `user_kbdvf8d6` (Tokyo), also rank in the top 10% per post; they are the genuine small-audience outperformers worth a closer look.

## 5. Summary

| Question | Literal answer | Evidence | Conclusion |
|---|---|---|---|
| E3 | Instagram, 4,040.0 | 2.2% spread, p = 0.56, detectable gap 3.7% | Any platform effect is below 4% |
| M4 | Instagram, 4,145.2 (+4.4% vs smaller accounts) | Holm-adjusted p > 0.05; rankings reshuffle | No stable platform advantage for large accounts |
| H4 | 16 users | ~13 expected by chance; 14 volume-driven | 2 genuine per-post outperformers |

Deliverables: `Entropy_Phase2_SQL_Queries.pdf`, `images/Entropy_{E3,M4,H4}_output.jpeg`, `Entropy_Phase2_Logic_Explanation.pdf`, `Entropy_Phase2_Insight_Report.pdf`.

In [14]:
con.close()